In [1]:
from autograd.core.engine import Value
from autograd.core.ui import show

#### Example: Neural Network

In [2]:
from random import uniform
from uuid import uuid4

def _indent(text, prefix="  "):
    return "\n".join(prefix + line for line in text.splitlines())

class Module:
    def __init__(self):
        pass

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0
    
    def parameters(self):
        return []

class Neuron(Module):
    def __init__(self, n_in):
        self.w = [Value(uniform(-1, 1), label=f'[w]{str(uuid4())[:8]}') for _ in range(n_in)]
        self.b = Value(uniform(-1, 1), label=f'[b]{str(uuid4())[:8]}')

    def forward(self, xs):
        return sum((x * w for w, x in zip(self.w, xs)), self.b).tanh()

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        n_w = ", ".join(f"{w.data:.4f}" for w in self.w)
        n_b = f"{self.b.data:.4f}"
        return f"Neuron[w=[{n_w}] | b={n_b}]"

class Layer(Module):
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]

    def forward(self, xs):
        out = [n(xs) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        body = "\n".join(_indent(repr(n)) for n in self.neurons)
        return f"Layer[\n{body}\n]"

class MLP(Module):
    def __init__(self, n_in, n_out):
        size = [n_in] + n_out
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(size) - 1)]

    def forward(self, xs):
        for l in self.layers:
            xs = l(xs)
        return xs
    
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

    def __repr__(self):
        body = "\n".join(_indent(repr(l)) for l in self.layers)
        return f"MLP[\n{body}\n]"

In [3]:
nn = MLP(3, [16, 16, 1])
nn.parameters()[:10], len(nn.parameters())

([Value([w]a18babee | data=0.006503606624514324 | grad=0.0),
  Value([w]a4b77131 | data=0.33868091389573474 | grad=0.0),
  Value([w]129318e4 | data=0.885641874246224 | grad=0.0),
  Value([b]71917f33 | data=0.20360993866256583 | grad=0.0),
  Value([w]1c2da251 | data=-0.7805864977889443 | grad=0.0),
  Value([w]97ce8dea | data=-0.4990179604042022 | grad=0.0),
  Value([w]52ea5100 | data=0.7254856800804133 | grad=0.0),
  Value([b]6065c2fc | data=-0.7223932396573065 | grad=0.0),
  Value([w]a383e504 | data=-0.8817738553669341 | grad=0.0),
  Value([w]0dc1cbe4 | data=-0.6311867875114854 | grad=0.0)],
 353)

In [4]:
X = [
    [ 1.0,  5.3,  4.1],
    [ 3.5,  3.1, -1.5],
    [-1.4,  4.3,  1.9],
    [-0.5, -0.8,  3.1],
]
y = [
     1.0,
    -1.0,
    -1.0,
     1.0
]

learning_rate = 0.05

for epoch in range(50):
    loss = 0.0
    
    # forward pass
    y_hat = [nn(xs) for xs in X]
    loss = sum(((y_true - y_pred)**2 for y_pred, y_true in zip(y_hat, y)))
    if epoch % 5 == 0:
        print("\nComparison:")
        for y_pred, y_true in zip(y_hat, y):
            print(f"y_pred={y_pred.data:.4f}, y_true={y_true:.4f}")
        print()

    # backward pass
    nn.zero_grad()
    loss.backward()

    # optimize
    for p in nn.parameters():
        p.data -= learning_rate * p.grad
    
    print(f"{epoch + 1}: loss={loss.data:.4f}")


Comparison:
y_pred=-0.5253, y_true=1.0000
y_pred=0.3766, y_true=-1.0000
y_pred=-0.4193, y_true=-1.0000
y_pred=0.3314, y_true=1.0000

1: loss=5.0058
2: loss=1.0614
3: loss=0.2182
4: loss=0.2357
5: loss=3.4433

Comparison:
y_pred=-0.1441, y_true=1.0000
y_pred=-0.9849, y_true=-1.0000
y_pred=-0.9835, y_true=-1.0000
y_pred=0.9984, y_true=1.0000

6: loss=1.3095
7: loss=4.0207
8: loss=3.9926
9: loss=3.9895
10: loss=3.9873

Comparison:
y_pred=0.9847, y_true=1.0000
y_pred=-0.9477, y_true=-1.0000
y_pred=0.9956, y_true=-1.0000
y_pred=0.9991, y_true=1.0000

11: loss=3.9854
12: loss=3.9836
13: loss=3.9817
14: loss=3.9797
15: loss=3.9775

Comparison:
y_pred=0.9776, y_true=1.0000
y_pred=-0.9655, y_true=-1.0000
y_pred=0.9933, y_true=-1.0000
y_pred=0.9990, y_true=1.0000

16: loss=3.9749
17: loss=3.9719
18: loss=3.9683
19: loss=3.9639
20: loss=3.9582

Comparison:
y_pred=0.9603, y_true=1.0000
y_pred=-0.9770, y_true=-1.0000
y_pred=0.9871, y_true=-1.0000
y_pred=0.9989, y_true=1.0000

21: loss=3.9508
22: l

In [5]:
nn

MLP[
  Layer[
    Neuron[w=[-0.1309, 0.2133, 0.9448] | b=0.1638]
    Neuron[w=[-1.2719, -1.0027, 0.0769] | b=-0.7846]
    Neuron[w=[-0.8757, -0.6675, -0.2181] | b=-0.2875]
    Neuron[w=[-0.4477, 0.6588, -1.2892] | b=-0.1410]
    Neuron[w=[0.6018, -0.8642, 0.5375] | b=-0.1347]
    Neuron[w=[0.2406, -0.4367, -0.9795] | b=-0.7985]
    Neuron[w=[0.5104, 0.6254, -0.2160] | b=0.6683]
    Neuron[w=[-1.0704, 0.8234, 0.4008] | b=0.0516]
    Neuron[w=[-0.6998, -0.1928, -0.6550] | b=-0.2770]
    Neuron[w=[-0.9076, 0.7664, 0.8700] | b=-0.9957]
    Neuron[w=[-0.0992, -1.1614, -0.7321] | b=0.6860]
    Neuron[w=[-0.2367, 0.4908, 0.5063] | b=0.8052]
    Neuron[w=[0.7924, -0.4875, 1.1934] | b=-0.8097]
    Neuron[w=[0.8020, 0.5016, -0.5910] | b=0.4922]
    Neuron[w=[-0.9225, -0.8306, 0.9457] | b=-0.2862]
    Neuron[w=[0.3279, 0.4658, -0.3305] | b=0.4020]
  ]
  Layer[
    Neuron[w=[-0.1431, 0.0479, 0.2407, 0.9704, 0.4101, 0.4581, -0.3869, -0.6477, -0.2326, 0.8750, 0.9508, 0.7878, -0.1758, 0.8122, 0.6342,